# Part 1: Filtering Pandas with Strings

## Import Modules

In [ ]:
# Import standard data science libraries
import numpy as np
import pandas as pd

## Review of DataFrames

Let's create the baseball related DataFrame that we worked with last week.

In [ ]:
# Create a standard Python dictionary with lists of equal length
baseball_dict = {'City': ['Pittsburgh', 'Cincinatti', 'Chicago', 'St. Louis', 'Milwaukee'],
                 'Team': ['Pirates', 'Reds', 'Cubs', 'Cardinals', 'Brewers'],
                 'Division': 5 * ['Central'],
                 'League': 5 * ['NL']}

In [ ]:
# Convert dictionary to DataFrame, specifying the exact column order
baseball_df = pd.DataFrame( baseball_dict,
                            columns=['League', 'Division', 'City', 'Team'])

In [ ]:
baseball_df

Add a column for the number of games back.

In [ ]:
# Assigning a new Pandas Series to a new column name
baseball_df['games_back'] = pd.Series( [31.5, 27.5, 22.5, 0, 7.5],
                                       index=baseball_df.index )

Sort by the `games_back` column. Ignore the index, and modify in place!

In [ ]:
# Sort values ascending, resetting the index so it flows 0, 1, 2... sequentially
baseball_df.sort_values( ['games_back'], ignore_index=True, inplace=True)

Add two more columns that have values which change down the rows.

In [ ]:
# Adding 'wins' and 'losses' using Pandas Series mapping
baseball_df['wins'] = pd.Series([87, 79, 64, 59, 55],
                                index=baseball_df.index)

baseball_df['losses'] = pd.Series([63, 70, 85, 90, 94],
                                  index=baseball_df.index)

Lastly, add a column with a constant value down all rows.

In [ ]:
# Broadcasting a scalar value to an entire column
baseball_df['season'] = 2022
baseball_df

## Filter rows

Filtering refers to SELECTING rows based on CONDITIONAL TESTS.

In [ ]:
# Isolate rows where wins exceed 65, keep all columns
baseball_df.loc[ baseball_df.wins > 65, : ]

In [ ]:
# Exact string matching
baseball_df.loc[ baseball_df.Team == 'Pirates', : ]

We also saw how to use the OR operator, `|`, to find all rows where the value equals A or B.

Or, the value is ONE OF those presented.

In [ ]:
# Using the bitwise OR '|' operator. Note the mandatory parentheses around each condition!
baseball_df.loc[ (baseball_df.Team == 'Cardinals') | (baseball_df.Team == 'Brewers'), : ]

The `==` operator combined with `|` operator is correct to use...but it does not SCALE well!

For example, if we needed to check for 10 possible values...we would need to type in 10 different conditions!!!

Instead, we can use the `.isin()` method to streamline the `|` operator!

In [ ]:
# Passing a list of targets to .isin() is much cleaner
baseball_df.loc[ baseball_df.Team.isin(['Cardinals', 'Brewers']), :]

In [ ]:
# Easily scalable to 3+ targets
baseball_df.loc[ baseball_df.Team.isin(['Cardinals', 'Brewers', 'Pirates']), :]

The `.isin()` method can also be applied to numbers.

In [ ]:
baseball_df.loc[ baseball_df.losses.isin([70, 90]), :]

The `.isin()` operator is especially useful when passing dynamic lists as variables!

In [ ]:
# Store targets in a list variable
top_teams = baseball_df.loc[ baseball_df.games_back < 10, 'Team'].copy().tolist()
top_teams

In [ ]:
# Use the variable inside .isin()
baseball_df.loc[ baseball_df.Team.isin( top_teams ), : ]

## String pattern matching

In [ ]:
baseball_df.loc[ baseball_df.City == 'Pittsburgh', : ]

But what if I didn't feel like typing out the whole string for `'Pittsburgh'`?

In [ ]:
# This returns empty because 'Pitt' is NOT exactly equal to 'Pittsburgh'
baseball_df.loc[ baseball_df.City == 'Pitt', : ]

What if we had a typo?

In [ ]:
# This fails to find the target because of exact matching constraints
baseball_df.loc[ baseball_df.City == 'Pittsburg', :]

Instead, we could instead focus on a PATTERN. The `.str.contains()` method searches for a PATTERN **WITHIN** the string!

In [ ]:
# Returns a boolean Series (True if 'Pitt' is found anywhere in the string)
baseball_df.City.str.contains('Pitt')

In [ ]:
# Use the boolean Series as a filter
baseball_df.loc[ baseball_df.City.str.contains('Pitt'), : ]

We can even apply the PATTERN search to a single character!

In [ ]:
baseball_df.loc[ baseball_df.City.str.contains('P'), :]

But...be CAREFUL! If the PATTERN is TOO SHORT...it will not uniquely identify the string you are looking for!

In [ ]:
# 'C' exists in both 'Chicago' and 'Cincinatti'
baseball_df.loc[ baseball_df.City.str.contains('C'), : ]

The `.str.contains()` method is very helpful when EXPLORING data!

I particularly like to use it to search for non-letter characters.

To find a period in a string we need to search for the pattern `\\.`. (Because a regular dot `.` means "anything" in regular expressions!)

In [ ]:
baseball_df.loc[ baseball_df.City.str.contains( '\\.' ), : ]

We can even search for a WHITE SPACE.

In [ ]:
baseball_df.loc[ baseball_df.City.str.contains( ' ' ), :]

There are many more STRING METHODS available. Many of the Pandas `.str.` methods are consistent with the base Python string methods.

In [ ]:
# See all methods available under the .str accessor
dir( baseball_df.City.str )

---
# Part 2: Read data into Pandas (I/O)

Let's look at loading files from your local machine and from URLs.

In [ ]:
# OS library helps check directories to ensure our files exist
import os
os.listdir()

## Read Excel

In [ ]:
!pip install openpyxl --break-system-packages

In [ ]:
# Requires 'openpyxl' dependency
df0 = pd.read_excel( 'data/Excel_Example_Data.xlsx' )
df0.head()

We can force one of the columns to become the actual `.index` attribute when the data is read.

In [ ]:
# Tell Pandas to use the 0th column (Column A) as the row index
df0_b = pd.read_excel( 'data/Excel_Example_Data.xlsx', index_col=0 )
df0_b.head()

In [ ]:
df0_b.index

We do not just need the zeroth column to be the attribute. It can be any column!

In [ ]:
# Using column E (index 4) as the index
df0_c = pd.read_excel( 'data/Excel_Example_Data.xlsx', index_col=4 )
df0_c.head()

## Headers or Column names

By default, the `pd.read_*` family of functions ASSUMES the TOP row is the HEADER row!!!

The top row therefore does NOT contain VALUES!!! Instead, it is assumed the TOP ROW contains the COLUMN NAMES!

But, let's see what happens if we work with a dataset or a SHEET within an Excel workbook that does NOT use a header row!

When you know there is NO HEADER...then the `header` argument must be set to `None`.

In [ ]:
# Pass None to header to prevent Pandas from eating the first data row
df0_no_names = pd.read_excel( 'data/Excel_Example_Data.xlsx', sheet_name='no_headers', header=None )
df0_no_names.head()

If there is no header row, Pandas uses integers (0,1,2..) for columns. We can use the `names` argument to NAME the columns!

In [ ]:
df0_no_names_2 = pd.read_excel( 'data/Excel_Example_Data.xlsx', 
                                sheet_name='no_headers',
                                header=None,
                                names=df0.columns)
df0_no_names_2.head()

## Read CSV

CSV files have the extension `.csv`. Unlike Excel workbooks they contain a single spread sheet rather than multiple sheets.

`pd.read_csv()` has nearly all the same arguments as `pd.read_excel()`!

In [ ]:
# Example of limiting rows read in for massive files
pd.read_csv( 'data/Example_A.csv', nrows=3 )

In [ ]:
# Example of skipping the first 2 rows (be careful, this will drop headers too!)
pd.read_csv( 'data/Example_A.csv', skiprows=2 ).head()

In [ ]:
dfA = pd.read_csv( 'data/Example_A.csv' )
dfB = pd.read_csv( 'data/Example_B.csv' )
dfC = pd.read_csv( 'data/Example_C.csv' )

## Read or download from a website

The data might be located at a web address or URL instead of locally on your computer.

In [ ]:
gap_url = 'https://raw.githubusercontent.com/chendaniely/pandas_for_everyone/master/data/gapminder.tsv'

To read in the data we provide the URL string instead of a local file path!

Because we are reading a `.tsv` (TAB separated value) rather than a `.csv` (COMMA separated)...we need to change the `sep` argument to `\t`.

In [ ]:
gap_df = pd.read_csv( gap_url, sep = '\t' )
gap_df.head()

---
# Part 3: Combine DataFrames (Concatenation)


In [ ]:
# Load example data sets to manipulate
dfA0 = pd.read_csv( 'data/Example_A.csv' )
dfA0['attempt'] = 0

dfA1 = pd.read_csv( 'data/Example_A.csv' )
dfA1['attempt'] = 1

## Vertically Concatenate

Vertically combining means we STACK the objects on top of each other. The default `axis` is `axis=0`.

In [ ]:
pd.concat( [dfA0, dfA1] )

Look closely at the `.index` attribute of the COMBINED VERTICALLY STACKED DataFrames! 
By default, the `.index` attribute is allowed to repeat. 

Ignoring the index allows each stacked row to receive a unique sequential number!

In [ ]:
# Re-index starting from 0 to N
pd.concat( [dfA0, dfA1], ignore_index=True)

## Horizontal Concatenation

BINDING columns together side-by-side!

If we change `axis` to `axis=1` then the two DataFrames will be combined HORIZONTALLY!!!!!

In [ ]:
pd.concat( [dfA0, dfA1], axis=1 ).head()

In [ ]:
pd.concat( [dfA0, dfA1], axis=1 ).head().loc[:, 'attempt']

**WARNING:** The column names are NO LONGER UNIQUE!!!!

I really dislike that Pandas allows combining DataFrames horizontally even if they have the SAME COLUMN NAMES. This creates massive filtering bugs down the road.

The point of horizontally combining is to bring together DIFFERENT columns that have the SAME number of rows (e.g. binding Features matrix to Target array in Machine Learning).

---
# Part 4: Begin Exploring Data by Summarizing Pandas Series

We will explore data before training predictive models. This process is known as Exploratory Data Analysis (EDA).

An important aspect of EDA is knowing how to calculate SUMMARY STATISTICS.

## Review NumPy summary methods vs Pandas

Let's create a list of integers and then convert that list to a 1D NumPy array.

In [ ]:
my_list = [10, 20, 30, 40, 50, 60, 70, 80]
my_array = np.array( my_list )
my_series = pd.Series( my_list )

Most of the Pandas Series summary methods work very similarly to their NumPy counterparts. 
BUT...LOOK CLOSELY...at the VARIANCE!!!!

In [ ]:
# Pandas Variance
my_series.var()

In [ ]:
# Numpy Variance
my_array.var()

Why are they different?

Pandas CORRECTLY sets Delta Degrees of Freedom to `ddof=1` when the variance or standard deviation are calculated!!! This means Pandas calculates the UNBIASED (sample) estimate to variance and standard deviation!

NumPy assumes `ddof=0` (Population variance). To make them match, you must explicitly tell NumPy to use a sample:

In [ ]:
my_array.var(ddof=1)

### Unique values & Counts

Knowing the number of unique values is especially important for CATEGORICAL or STRING variables!

In [ ]:
my_series_b = pd.Series( ['A', 'A', 'A', 'B', 'B', 'B', 'C', 'D', 'D'])
# select count(distinct col1) from my_series_b
# Return the raw count of distinct elements
my_series_b.nunique()

The number of unique values does NOT need to equal the number of elements or SIZE!!!!

My favorite Pandas method focuses on dealing with unique values!!! Often times we want to COUNT the number of times a unique value occurs!

In [ ]:
# Get frequency counts of each unique category
my_series_b.value_counts()

If you just want the unique values (array format), then you can use the `.unique()` method.

In [ ]:
my_series_b.unique()

## Summarize individual columns within DataFrames

This is to reinforce the fact that COLUMNS are really just Pandas Series living within a DataFrame.

In [ ]:
df = pd.read_csv('data/joined_data.csv')
df.head()

You can apply summary methods like `.mean()`, `.std()`, and `.sem()` to any numeric column using dot notation or bracket notation!

In [ ]:
df.F.mean()

In [ ]:
df['F'].mean()

We can also calculate the STANDARD ERROR ON THE MEAN (SEM)!!!

In [ ]:
# Manual calculation (Std Dev / square root of sample size)
df.F.std() / np.sqrt( df.F.size )

In [ ]:
# Built-in Pandas method for Standard Error of the Mean
df.F.sem()